# ridge 모델 검증: 신검단중앙역 금강펜테리움센트럴파크

(1) 라이브러리 임포트 및 한글 폰트 설정

In [9]:
import pandas as pd
import matplotlib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# 한글 폰트 설정
matplotlib.rc('font', family='Malgun Gothic')
matplotlib.rc('axes', unicode_minus=False)

(2) 데이터 불러오기

In [10]:
train_folder = r"C:\Users\user22\Desktop\아파트 값 예측"
test_folder = r"C:\Users\user22\Desktop\아파트 값 예측"

df_all = pd.read_csv(train_folder + r"\new_city2.csv", encoding='utf-8-sig')
df_train = df_all[df_all['도시명'].isin(['광교', '판교', '운정'])].copy()
df_test = pd.read_csv(test_folder + r"\geomdan_merged2.csv", encoding='utf-8-sig')

(3) 전처리 (거래금액, m2당 가격, 필터링)

In [11]:
# 거래금액
df_train['거래금액(만원)'] = df_train['거래금액(만원)'].str.replace(',', '').astype(int)
df_test['거래금액(만원)'] = df_test['거래금액(만원)'].str.replace(',', '').astype(int)

# 전용면적당 가격 계산
df_train['m2당가격'] = df_train['거래금액(만원)'] / df_train['전용면적(㎡)']
df_test['m2당가격'] = df_test['거래금액(만원)'] / df_test['전용면적(㎡)']

# 발표후경과년수 3년 미만 제거
df_train = df_train[df_train['발표후경과년수'] >= 3]
df_test = df_test[df_test['발표후경과년수'] >= 3]

# 2022년 이하만 훈련에 사용
df_train = df_train[df_train['계약연도'] <= 2022]
df_geomdan_train = df_test[df_test['계약연도'] <= 2022].copy()
df_train = pd.concat([df_train, df_geomdan_train], ignore_index=True)

(4) 이상치 제거

In [12]:
# z-score
df_train = df_train[df_train['전용면적(㎡)'] >= 33]

mean = df_train['m2당가격'].mean()
std = df_train['m2당가격'].std()
z_scores = (df_train['m2당가격'] - mean) / std
df_train = df_train[z_scores.abs() <= 2]

print(f"훈련 데이터: {len(df_train)}건")

훈련 데이터: 48440건


(5) 독립변수/타겟 설정 및 표준화

In [13]:
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

df_train = df_train.dropna(subset=features + ['m2당가격'])

train_input = df_train[features]
train_target = df_train['m2당가격']

ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)

(6) 모델 학습

In [14]:
ridge_best = Ridge(alpha=0.001)
ridge_best.fit(train_scaled, train_target)

print(f"훈련 R²: {ridge_best.score(train_scaled, train_target):.4f}")

훈련 R²: 0.8124


(7) 검증 데이터 설정

In [15]:
# 신검단중앙역금강펜테리움센트럴파크
금강_기본 = {
    '건축년도': 2025,
    '층': 12,
    '지하철호선개수': 3,
    '기차역까지의거리': 16.7135,
    '가장 가까운 지하철역까지의 거리': 0.894,
    '가장 가까운 IC와의 거리': 5.804,
    '발표후경과년수': 20,
    'CPI': 116.61,
    '계약연도': 2026,
    '서울도심거리': 25.136,
    '단지별_세대수': 1049.0,
    '도시별_세대수': 60833.0,
}

면적_검증 = 86.23   # 2026년 평균 전용면적
실제_거래금액 = 52093  # 2026년 평균 실제 거래금액 (38건 평균)

(8) 예측 및 결과

In [16]:
# 실제값과 비교해 오차 및 오차율 출력
입력 = pd.DataFrame([금강_기본])[features]
입력_scaled = ss.transform(입력)
예측_m2 = ridge_best.predict(입력_scaled)[0]
예측_거래금액 = int(예측_m2 * 면적_검증)
오차율 = abs(예측_거래금액 - 실제_거래금액) / 실제_거래금액 * 100

print(f"\n{'='*60}")
print(f"모델 검증: 신검단중앙역금강펜테리움센트럴파크 (2026년, 38건 평균)")
print(f"{'='*60}")
print(f"실제 거래금액 (평균): {실제_거래금액:,}만원 ({실제_거래금액/10000:.1f}억)")
print(f"예측 거래금액       : {예측_거래금액:,}만원 ({예측_거래금액/10000:.1f}억)")
print(f"오차               : {예측_거래금액 - 실제_거래금액:+,}만원")
print(f"오차율             : {오차율:.1f}%")


모델 검증: 신검단중앙역금강펜테리움센트럴파크 (2026년, 38건 평균)
실제 거래금액 (평균): 52,093만원 (5.2억)
예측 거래금액       : 54,703만원 (5.5억)
오차               : +2,610만원
오차율             : 5.0%
